# San Diego Property Management Companies: Clean & Enrich

Merges property listings from two sources:
1. **apartments.com** PMC directory scrape (75 companies, ~477 listings)
2. **Direct company website scrapes** (Greystar, CONAM, Irvine Company)

Deduplicates across sources, joins to SANDAG parcel data for unit counts,
and produces the definitive PM company to building to unit count mapping.

## 1. Load Raw Scrape Data

In [1]:
import json
from collections import defaultdict
from pathlib import Path

import polars as pl

DATA_DIR = Path("data")
PARCEL_DIR = Path("..") / "sandag" / "data"

# Load PMC directory for company names
with open(DATA_DIR / "pmc_directory.json") as f:
    pmc_dir = json.load(f)
slug_to_name = {p["slug"]: p["name"] for p in pmc_dir}

# Load raw progress and deduplicate
with open(DATA_DIR / "pmc_progress.jsonl") as f:
    raw_lines = [json.loads(l) for l in f if l.strip()]

seen_ids: set[str] = set()
rows: list[dict] = []
for record in raw_lines:
    slug = record["pmc_slug"]
    for listing in record["listings"]:
        lid = listing["listing_id"]
        if lid in seen_ids:
            continue
        seen_ids.add(lid)
        rows.append({
            "pmc_name": slug_to_name.get(slug, slug),
            "pmc_slug": slug,
            **listing,
        })

raw = pl.DataFrame(rows)
print(f"Raw deduplicated listings: {raw.shape[0]:,}")
print(f"PMCs represented: {raw['pmc_slug'].n_unique()}")
print(f"Columns: {raw.columns}")
raw.head(5)

Raw deduplicated listings: 477
PMCs represented: 74
Columns: ['pmc_name', 'pmc_slug', 'listing_id', 'name', 'street_address', 'full_address', 'unit_number', 'url']


pmc_name,pmc_slug,listing_id,name,street_address,full_address,unit_number,url
str,str,str,str,str,str,str,str
"""Greystar""","""greystar""","""3sx8thg""","""The Lindley""","""1331 Columbia St""","""1331 Columbia St, San Diego, C…","""""","""https://www.apartments.com/the…"
"""Greystar""","""greystar""","""te75978""","""Silo""","""7231 Colchester Ct""","""7231 Colchester Ct, San Diego,…","""""","""https://www.apartments.com/sil…"
"""Greystar""","""greystar""","""v0mjk9d""","""Folia""","""12520 Camino del Sur""","""12520 Camino del Sur, San Dieg…","""""","""https://www.apartments.com/fol…"
"""Greystar""","""greystar""","""t0kkk4c""","""2911 Adams""","""2911 Adams Ave""","""2911 Adams Ave, San Diego, CA …","""""","""https://www.apartments.com/291…"
"""Greystar""","""greystar""","""tensts9""","""Alexan Camellia""","""4888 Convoy St""","""4888 Convoy St, San Diego, CA …","""""","""https://www.apartments.com/ale…"


## 2. Load Direct Company Scrapes

Supplements apartments.com data with properties scraped directly from
company websites. These catch properties that apartments.com missed,
especially in suburban cities (Chula Vista, Escondido, Carlsbad, etc.).

In [2]:
# Load direct scrapes: Greystar, CONAM, Irvine Company
direct = pl.read_csv(DATA_DIR / "pmc_direct_properties.csv", infer_schema_length=1000)

# Load additional direct scrapes: Good Life PM, etc.
additional_path = DATA_DIR / "pmc_direct_additional.csv"
if additional_path.exists():
    additional = pl.read_csv(additional_path, infer_schema_length=1000)
    # Clean: drop junk rows (bad address parses)
    additional = additional.filter(
        pl.col("address").is_not_null()
        & (pl.col("address") != "")
        & ~pl.col("address").str.contains(r"(?i)office|suite \d{3}$|calculator")
        & (pl.col("address").str.len_chars() < 50)
    )
    direct = pl.concat([direct, additional], how="diagonal_relaxed")
    print(f"Additional direct scrapes: {additional.shape[0]:,} properties")

# Drop rows with no address
direct = direct.filter(pl.col("address").is_not_null() & (pl.col("address") != ""))

direct = direct.with_columns(
    pl.col("zip").cast(pl.String).str.slice(0, 5).alias("zip5"),
    pl.col("address").str.to_uppercase().str.strip_chars().str.replace_all(r"\s+", " ").alias("street_upper"),
)

print(f"Direct scrape properties (total): {direct.shape[0]:,}")
print(direct["pmc_name"].value_counts().sort("count", descending=True))

Additional direct scrapes: 26 properties
Direct scrape properties (total): 264
shape: (3, 2)
┌───────────────────────────────┬───────┐
│ pmc_name                      ┆ count │
│ ---                           ┆ ---   │
│ str                           ┆ u32   │
╞═══════════════════════════════╪═══════╡
│ CONAM Management Corporation  ┆ 124   │
│ Greystar                      ┆ 114   │
│ Good Life Property Management ┆ 26    │
└───────────────────────────────┴───────┘


## 3. Merge & Deduplicate

Combine apartments.com and direct-scrape listings into a single dataset.
Deduplicates by normalized address (street number + street name) so each
physical building appears only once. When a property exists in both sources,
the apartments.com record is preferred (has listing ID and URL).

In [3]:
# Use parse_apartments_street (defined in next cell) for dedup too.
# Inline the same normalization here to ensure consistent keys.
def make_dedup_key(addr_col: str) -> pl.Expr:
    """Create a dedup key: 'NUM|NORMALIZED_STREET'."""
    num = pl.col(addr_col).str.to_uppercase().str.extract(r"^(\d+)", 1)
    street = (
        pl.col(addr_col).str.to_uppercase().str.strip_chars()
        .str.replace_all(r"\.", "")  # strip periods
        .str.replace(r"\s+-\s+.*$", "")  # strip notes like " - Leasing Office East"
        .str.replace(r"^\d+(?:-?[A-Z])?\s+", "")  # strip number
        .str.replace(r"^\d+-\d+\s+", "")  # strip ranges
        .str.replace(r"\s+(?:#|UNIT|STE|SUITE)\s*\S+\s*$", "")  # strip unit
        # Strip trailing direction before suffix
        .str.replace(r"\s+(?:NORTH|SOUTH|EAST|WEST|NE|NW|SE|SW|N|S|E|W)\s*$", "")
        # Strip suffixes (3 passes)
        .str.replace(r"\s+(STREET|AVENUE|BOULEVARD|COURT|DRIVE|PLACE|ROAD|LANE|CIRCLE|TERRACE|PARKWAY|HIGHWAY|POINT|AVE?|AV|BLVD?|BL|ST|CT|DR|PL|WAY|RD|LN|CIR|TER|TERR|PKWY|PKY|HWY|CV|WY|LOOP|PT|SQ|ALY|WALK|GLN|RUN|XING|TRACE|KNOLL|VW|SPUR|PATH|CRES|VIA)\s*$", "")
        .str.replace(r"\s+(STREET|AVENUE|BOULEVARD|COURT|DRIVE|PLACE|ROAD|LANE|CIRCLE|TERRACE|PARKWAY|HIGHWAY|POINT|AVE?|AV|BLVD?|BL|ST|CT|DR|PL|WAY|RD|LN|CIR|TER|TERR|PKWY|PKY|HWY|CV|WY|LOOP|PT|SQ|ALY|WALK|GLN|RUN|XING|TRACE|KNOLL|VW|SPUR|PATH|CRES|VIA)\s*$", "")
        .str.replace(r"\s+(STREET|AVENUE|BOULEVARD|COURT|DRIVE|PLACE|ROAD|LANE|CIRCLE|TERRACE|PARKWAY|HIGHWAY|POINT|AVE?|AV|BLVD?|BL|ST|CT|DR|PL|WAY|RD|LN|CIR|TER|TERR|PKWY|PKY|HWY|CV|WY|LOOP|PT|SQ|ALY|WALK|GLN|RUN|XING|TRACE|KNOLL|VW|SPUR|PATH|CRES|VIA)\s*$", "")
        # Strip trailing direction again
        .str.replace(r"\s+(?:NORTH|SOUTH|EAST|WEST|NE|NW|SE|SW|N|S|E|W)\s*$", "")
        # Strip leading direction
        .str.replace(r"^(?:NORTH|SOUTH|EAST|WEST|NE|NW|SE|SW|N|S|E|W)\s+", "")
        # Ordinals
        .str.replace(r"\bFIRST\b", "1ST").str.replace(r"\bSECOND\b", "2ND")
        .str.replace(r"\bTHIRD\b", "3RD").str.replace(r"\bFOURTH\b", "4TH")
        .str.replace(r"\bFIFTH\b", "5TH").str.replace(r"\bSIXTH\b", "6TH")
        .str.replace(r"\bSEVENTH\b", "7TH").str.replace(r"\bEIGHTH\b", "8TH")
        .str.replace(r"\bNINTH\b", "9TH").str.replace(r"\bTENTH\b", "10TH")
        # MOUNT -> MT
        .str.replace_all(r"\bMOUNT\s+", "MT ")
        .str.replace("MAHAILA", "MAHALIA")  # known typo
        .str.replace(r"^0+(\d)", "$1")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )
    return num + "|" + street


# Known address typos in source data
TYPO_FIXES = {
    "MAHAILA": "MAHALIA",
}


# Add dedup keys to apartments.com data
apt_keyed = raw.with_columns(
    pl.col("full_address").str.extract(r",\s*([^,]+),\s*[A-Z]{2}\s+\d{5}", 1).alias("city"),
    pl.col("full_address").str.extract(r"(\d{5})", 1).alias("zip5"),
    make_dedup_key("street_address").alias("dedup_key"),
    pl.lit("apartments.com").alias("source"),
)

# Add dedup keys and synthetic listing_ids to direct scrape
direct_keyed = direct.with_columns(
    make_dedup_key("address").alias("dedup_key"),
    pl.lit("direct").alias("source"),
    (pl.lit("direct_") + pl.arange(0, direct.shape[0]).cast(pl.String)).alias("listing_id"),
    pl.lit(None).cast(pl.String).alias("full_address"),
    pl.lit(None).cast(pl.String).alias("unit_number"),
    pl.lit(None).cast(pl.String).alias("url"),
    pl.lit(None).cast(pl.String).alias("pmc_slug"),
).rename({"address": "street_address"})

# Deduplicate: keep apartments.com version when both sources have it
apt_keys = set(apt_keyed["dedup_key"].drop_nulls().to_list())
direct_new = direct_keyed.filter(~pl.col("dedup_key").is_in(list(apt_keys)))

# Align columns and merge
common_cols = ["pmc_name", "listing_id", "name", "street_address",
               "city", "zip5", "source", "dedup_key"]

merged = pl.concat([
    apt_keyed.select([c for c in common_cols if c in apt_keyed.columns]),
    direct_new.select([c for c in common_cols if c in direct_new.columns]),
], how="diagonal_relaxed")

# Final dedup: within merged, remove rows with duplicate keys
merged = merged.unique(subset=["dedup_key"], keep="first")

# Fuzzy dedup for address ranges: "2412|ESCONDIDO" and "2414|ESCONDIDO" are the same building.
# Extract the street part of dedup_key and group by it. If two rows share the same
# street and their numbers are within 20, keep only the first.
merged = merged.with_columns(
    pl.col("dedup_key").str.extract(r"^(\d+)\|", 1).cast(pl.Int64, strict=False).alias("_dk_num"),
    pl.col("dedup_key").str.extract(r"\|(.+)$", 1).alias("_dk_street"),
)
# Sort so apartments.com source comes first (preferred)
merged = merged.sort("source")
keep_ids: set[int] = set()
seen_streets: dict[str, list[tuple[int, int, str]]] = {}  # street -> [(num, row_idx, pmc)]
for idx, row in enumerate(merged.to_dicts()):
    street = row.get("_dk_street", "")
    num = row.get("_dk_num")
    if not street or num is None:
        keep_ids.add(idx)
        continue
    if street not in seen_streets:
        seen_streets[street] = []
    pmc = row.get("pmc_name", "")
    # Check if any existing entry is within 20 with the same PMC
    is_dupe = False
    for prev_num, prev_idx, prev_pmc in seen_streets[street]:
        if abs(num - prev_num) <= 20 and pmc == prev_pmc:
            is_dupe = True
            break
    if not is_dupe:
        keep_ids.add(idx)
        seen_streets[street].append((num, idx, pmc))

merged = merged.with_row_index("_idx").filter(pl.col("_idx").is_in(list(keep_ids))).drop("_idx", "_dk_num", "_dk_street")

print(f"apartments.com listings: {apt_keyed.shape[0]:,}")
print(f"Direct scrape listings:  {direct_keyed.shape[0]:,}")
print(f"New from direct scrape:  {direct_new.shape[0]:,}")
print(f"Merged (after dedup):   {merged.shape[0]:,}")
print()
print("Source breakdown:")
print(merged["source"].value_counts().sort("count", descending=True))
print()
print("New properties from direct scrape by PMC:")
print(direct_new["pmc_name"].value_counts().sort("count", descending=True))

apartments.com listings: 477
Direct scrape listings:  264
New from direct scrape:  181
Merged (after dedup):   639

Source breakdown:
shape: (2, 2)
┌────────────────┬───────┐
│ source         ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ apartments.com ┆ 465   │
│ direct         ┆ 174   │
└────────────────┴───────┘

New properties from direct scrape by PMC:
shape: (3, 2)
┌───────────────────────────────┬───────┐
│ pmc_name                      ┆ count │
│ ---                           ┆ ---   │
│ str                           ┆ u32   │
╞═══════════════════════════════╪═══════╡
│ CONAM Management Corporation  ┆ 115   │
│ Greystar                      ┆ 40    │
│ Good Life Property Management ┆ 26    │
└───────────────────────────────┴───────┘


## 4. Clean & Normalize Addresses

Parse addresses into components for parcel matching.

In [4]:
listings = merged.with_columns(
    pl.col("street_address")
    .str.to_uppercase()
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .alias("street_upper"),
)

print(f"Listings: {listings.shape[0]:,}")
print(f"\nCity distribution:")
print(listings["city"].value_counts().sort("count", descending=True).head(15))
print(f"\nSource distribution:")
print(listings["source"].value_counts().sort("count", descending=True))
print(f"\nMissing zip: {listings.filter(pl.col('zip5').is_null()).shape[0]}")

Listings: 639

City distribution:
shape: (15, 2)
┌───────────────┬───────┐
│ city          ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ San Diego     ┆ 379   │
│ Chula Vista   ┆ 61    │
│ La Mesa       ┆ 28    │
│ El Cajon      ┆ 25    │
│ Carlsbad      ┆ 17    │
│ …             ┆ …     │
│ National City ┆ 11    │
│ La Jolla      ┆ 7     │
│ Encinitas     ┆ 6     │
│ Lakeside      ┆ 5     │
│ Alpine        ┆ 5     │
└───────────────┴───────┘

Source distribution:
shape: (2, 2)
┌────────────────┬───────┐
│ source         ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ apartments.com ┆ 465   │
│ direct         ┆ 174   │
└────────────────┴───────┘

Missing zip: 0


## 5. PMC Summary

Properties per management company after merging all sources.

In [5]:
pmc_summary = (
    listings
    .group_by("pmc_name")
    .agg(
        pl.len().alias("properties"),
        pl.col("city").n_unique().alias("cities"),
    )
    .sort("properties", descending=True)
)

print(f"Property management companies: {pmc_summary.shape[0]}")
print(f"Total properties: {listings.shape[0]:,}")
print()
pmc_summary

Property management companies: 74
Total properties: 639



pmc_name,properties,cities
str,u32,u32
"""CONAM Management Corporation""",128,15
"""Greystar""",114,12
"""R.A. Snyder Properties, Inc.""",38,9
"""Sunrise Management""",29,6
"""Good Life Property Management""",26,9
…,…,…
"""Decron Properties""",1,1
"""Gables Residential""",1,1
"""Walsh & Company""",1,1


## 6. Join to SANDAG Parcel Data

Match addresses to parcel records for unit counts and assessed values.
Three-pass address matching:
1. Street number + street name + zip
2. Street number + street name (no zip)
3. Nearest address number on same street + zip (within 50)

In [6]:
parcels = pl.read_parquet(PARCEL_DIR / "parcels.parquet")
print(f"Parcels loaded: {parcels.shape[0]:,}")

LANDUSE_LABELS = {
    9: "Manufactured Home", 11: "Single Family", 12: "Duplex",
    13: "Multifamily 2-4", 14: "Apartment 5-15", 15: "Apartment 16-60",
    16: "Apartment 61+", 17: "Condo", 26: "Hotel/Motel",
}
label_df = pl.DataFrame({
    "ASR_LANDUS": list(LANDUSE_LABELS.keys()),
    "landuse_label": list(LANDUSE_LABELS.values()),
})
parcels = parcels.join(label_df, on="ASR_LANDUS", how="left")

Parcels loaded: 1,089,584


In [7]:
STREET_SUFFIXES = (
    r"STREET|AVENUE|BOULEVARD|COURT|DRIVE|PLACE|ROAD|LANE|CIRCLE|"
    r"TERRACE|PARKWAY|HIGHWAY|POINT|"
    r"AVE?|AV|BLVD?|BL|ST|CT|DR|PL|WAY|RD|LN|CIR|TER|TERR|"
    r"PKWY|PKY|HWY|CV|WY|LOOP|PT|SQ|ALY|WALK|GLN|RUN|XING|"
    r"TRACE|KNOLL|VW|SPUR|PATH|CRES|VIA"
)

DIRECTIONS = r"(?:NORTH|SOUTH|EAST|WEST|NE|NW|SE|SW|N|S|E|W)"


def normalize_street(s: pl.Expr) -> pl.Expr:
    """Normalize a street name for address matching."""
    return (
        s
        .str.to_uppercase()
        .str.strip_chars()
        .str.replace_all(r"\s+", " ")
        .str.replace(r"^0+(\d)", "$1")
        .str.replace_all(r"\bMT\.\s*", "MT ")
        .str.replace_all(r"\bST\.\s*", "SAINT ")
        .str.replace_all(r"[%#]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def parse_apartments_street(s: pl.Expr) -> pl.Expr:
    """Parse an apartments.com street address into the parcel street name.

    Apartments.com: "S MOLLISON AVE" or "EL CAJON BLVD"
    Parcels store:   SITUS_PRE_="S", SITUS_STRE="MOLLISON", SITUS_SUFF="AVE"

    Steps:
    1. Strip leading number ("1150 E ST" -> "E ST")
    2. Strip address ranges ("4747-4759 67TH ST" -> "67TH ST")
    3. Strip trailing street suffix ("67TH ST" -> "67TH")
    4. Strip leading directional prefix ("S MOLLISON" -> "MOLLISON")
    5. Normalize ordinal streets ("FOURTH" -> "4TH")
    """
    return (
        s
        .str.to_uppercase()
        .str.strip_chars()
        # Strip periods (DR. -> DR, AVE. -> AVE)
        .str.replace_all(r"\.", "")
        # Strip notes like " - Leasing Office East"
        .str.replace(r"\s+-\s+.*$", "")
        # Fix known typos
        .str.replace("MAHAILA", "MAHALIA")
        # Strip unit/suite numbers (#106, UNIT 100)
        .str.replace(r"\s+(?:#|UNIT|STE|SUITE)\s*\S+\s*$", "")
        # Strip leading street number (with optional letter suffix: "10802-B ")
        .str.replace(r"^\d+(?:-?[A-Z])?\s+", "")
        # Strip address ranges like "4747-4759 "
        .str.replace(r"^\d+-\d+\s+", "")
        # Strip trailing directional FIRST (HOTEL CIR N -> HOTEL CIR)
        .str.replace(rf"\s+{DIRECTIONS}\s*$", "")
        # Strip trailing suffix (run 3x for chains like "SANTA RITA DR E" -> DR stripped, then direction already gone)
        .str.replace(rf"\s+({STREET_SUFFIXES})\s*$", "")
        .str.replace(rf"\s+({STREET_SUFFIXES})\s*$", "")
        .str.replace(rf"\s+({STREET_SUFFIXES})\s*$", "")
        # Strip trailing directional again (in case suffix removal exposed one)
        .str.replace(rf"\s+{DIRECTIONS}\s*$", "")
        # Strip leading directional prefix (S MOLLISON -> MOLLISON)
        .str.replace(rf"^{DIRECTIONS}\s+", "")
        # Written-out ordinals to numeric (works anywhere in the string)
        .str.replace(r"\bFIRST\b", "1ST")
        .str.replace(r"\bSECOND\b", "2ND")
        .str.replace(r"\bTHIRD\b", "3RD")
        .str.replace(r"\bFOURTH\b", "4TH")
        .str.replace(r"\bFIFTH\b", "5TH")
        .str.replace(r"\bSIXTH\b", "6TH")
        .str.replace(r"\bSEVENTH\b", "7TH")
        .str.replace(r"\bEIGHTH\b", "8TH")
        .str.replace(r"\bNINTH\b", "9TH")
        .str.replace(r"\bTENTH\b", "10TH")
        # Normalize MOUNT -> MT (parcels use MT ETNA not MOUNT ETNA)
        .str.replace_all(r"\bMOUNT\s+", "MT ")
        # Strip leading zeros on numbered streets
        .str.replace(r"^0+(\d)", "$1")
        # Strip leading fractional part (1/2, 1/3 after number was removed)
        .str.replace(r"^\d/\d\s+", "")
        .str.replace(r"\s+\d/\d\s*", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


# Parse apartments.com addresses: extract number and street name
listings_parsed = listings.with_columns(
    # Street number: first number in the address, ignoring ranges
    pl.col("street_upper")
    .str.extract(r"^(\d+)", 1)
    .alias("join_num"),
    # Street name: normalized to match SITUS_STRE
    parse_apartments_street(pl.col("street_upper")).alias("join_street"),
    pl.col("zip5").alias("join_zip"),
    pl.col("city").str.to_uppercase().alias("city_upper"),
)

# Build parcel lookup tables
PARCEL_AGG = [
    pl.col("APN").first(),
    pl.col("ASR_LANDUS").first(),
    pl.col("UNITQTY").sum().alias("total_units"),
    pl.col("ASR_TOTAL").sum().alias("assessed_value"),
    pl.col("OWNEROCC").first(),
    pl.col("SITUS_COMM").first(),
    pl.col("landuse_label").first(),
    pl.len().alias("parcel_count"),
]

parcels_norm = (
    parcels
    .with_columns(
        pl.col("SITUS_ADDR").cast(pl.String).str.strip_chars().alias("join_num"),
        normalize_street(pl.col("SITUS_STRE")).alias("join_street"),
        pl.col("SITUS_ZIP").str.slice(0, 5).alias("join_zip"),
        pl.col("SITUS_COMM").str.to_uppercase().str.strip_chars().alias("SITUS_COMM_UPPER"),
    )
    .filter(
        (pl.col("join_num") != "0") & (pl.col("join_num") != "")
        & pl.col("join_street").is_not_null() & (pl.col("join_street") != "")
    )
)

# Pass 1: number + street + zip
pa1 = parcels_norm.filter(
    pl.col("join_zip").is_not_null() & (pl.col("join_zip") != "")
).group_by("join_num", "join_street", "join_zip").agg(*PARCEL_AGG)

# Pass 2: number + street + city (no zip)
pa2 = parcels_norm.group_by("join_num", "join_street", "SITUS_COMM_UPPER").agg(*PARCEL_AGG)

# Pass 3: nearest address on same street + zip
pa3 = (
    parcels_norm
    .with_columns(pl.col("join_num").cast(pl.Int64, strict=False).alias("num_int"))
    .filter(pl.col("num_int").is_not_null() & (pl.col("num_int") > 0))
    .filter(pl.col("join_zip").is_not_null() & (pl.col("join_zip") != ""))
    .group_by("join_street", "join_zip", "num_int")
    .agg(*PARCEL_AGG)
)

# --- Pass 1: exact match on number + street + zip ---
p1 = listings_parsed.join(pa1, on=["join_num", "join_street", "join_zip"], how="left")
matched_p1 = p1.filter(pl.col("APN").is_not_null()).with_columns(pl.lit(1).alias("_match_pass"))
unmatched_p1 = p1.filter(pl.col("APN").is_null()).select(listings_parsed.columns)

# --- Pass 2: number + street + city, drop zip ---
p2 = unmatched_p1.join(
    pa2,
    left_on=["join_num", "join_street", "city_upper"],
    right_on=["join_num", "join_street", "SITUS_COMM_UPPER"],
    how="left",
)
matched_p2 = p2.filter(pl.col("APN").is_not_null()).with_columns(pl.lit(2).alias("_match_pass"))
unmatched_p2 = p2.filter(pl.col("APN").is_null()).select(listings_parsed.columns)

# --- Pass 3: nearest address number on same street + zip ---
unmatched_int = unmatched_p2.with_columns(
    pl.col("join_num").cast(pl.Int64, strict=False).alias("listing_num_int")
).filter(pl.col("listing_num_int").is_not_null())

p3 = (
    unmatched_int
    .join(pa3, on=["join_street", "join_zip"], how="inner")
    .with_columns(
        (pl.col("listing_num_int") - pl.col("num_int")).abs().alias("addr_dist")
    )
    .filter(pl.col("addr_dist") <= 50)
    .sort("addr_dist")
    .group_by("listing_id")
    .first()
)
matched_p3 = p3.filter(pl.col("APN").is_not_null()).with_columns(pl.lit(3).alias("_match_pass"))

# Include ALL listings: matched get parcel data, unmatched get nulls
all_matched = pl.concat([matched_p1, matched_p2, matched_p3], how="diagonal_relaxed")
matched_ids = set(all_matched["listing_id"].drop_nulls().to_list()) if all_matched.shape[0] > 0 else set()

still_unmatched = listings_parsed.filter(
    ~pl.col("listing_id").is_in(list(matched_ids))
)

enriched = pl.concat(
    [all_matched, still_unmatched],
    how="diagonal_relaxed",
)
matched_count = enriched.filter(pl.col("APN").is_not_null()).shape[0]
total = listings.shape[0]
unmatched_final = total - matched_count

print(f"Address matching results:")
print(f"  Pass 1 (num+street+zip):  {matched_p1.shape[0]:>4}")
print(f"  Pass 2 (num+street+city): {matched_p2.shape[0]:>4}")
print(f"  Pass 3 (nearest, +/-50):  {matched_p3.shape[0]:>4}")
print(f"  Total matched:            {matched_count:>4} / {total} ({matched_count/total*100:.1f}%)")
print(f"  Unmatched:                {unmatched_final:>4}")

if unmatched_final > 0:
    still_unmatched = unmatched_p2.filter(
        ~pl.col("listing_id").is_in(matched_p3["listing_id"].to_list())
    ) if matched_p3.shape[0] > 0 else unmatched_p2
    print(f"\nSample unmatched addresses:")
    for row in still_unmatched.select("name", "street_upper", "join_num", "join_street", "join_zip").head(10).to_dicts():
        print(f"  {row['street_upper']:<40} -> num={row['join_num']} street={row['join_street']} zip={row['join_zip']}")

Address matching results:
  Pass 1 (num+street+zip):  388
  Pass 2 (num+street):       80
  Pass 3 (nearest, +/-20):  115
  Total matched:            583 / 639 (91.2%)
  Unmatched:                 56

Sample unmatched addresses:
  6555 AMBROSIA DR                         -> num=6555 street=AMBROSIA zip=92124
  11832 STONEY PEAK DR                     -> num=11832 street=STONEY PEAK zip=11832
  10360 MAYA LINDA RD                      -> num=10360 street=MAYA LINDA zip=10360
  245 BONITA GLEN DR                       -> num=245 street=BONITA zip=91910
  1630 PASEO MONTI                         -> num=1630 street=PASEO MONTI zip=91913
  45418 HOWE RD                            -> num=45418 street=HOWE zip=45418
  3950 MAHAILA AVE                         -> num=3950 street=MAHALIA zip=92122
  10977 PACIFIC POINT PL                   -> num=10977 street=PACIFIC zip=10977
  4200 BROOKE CT                           -> num=4200 street=BROOKE zip=92122
  5080 CAMINO DEL ARROYO                 

## 7. Final Dataset: PMC to Building to Units

In [8]:
# Select columns and deduplicate by APN
result_cols = [
    "pmc_name", "name", "street_address", "city", "zip5", "source",
    "APN", "total_units", "assessed_value", "landuse_label", "SITUS_COMM",
    "_match_pass",
]
result_cols = [c for c in result_cols if c in enriched.columns]

result = enriched.select(result_cols).sort("pmc_name", "name")

# Dedup by APN
matched = result.filter(pl.col("APN").is_not_null()).unique(subset=["APN"], keep="first")
unmatched = result.filter(pl.col("APN").is_null())
result = pl.concat([matched, unmatched]).sort("pmc_name", "name")

# Identify PMCs that are clearly multi-family operators (>= 5 properties)
pmc_property_counts = (
    result
    .group_by("pmc_name")
    .agg(pl.len().alias("pmc_total_properties"))
)
result = result.join(pmc_property_counts, on="pmc_name", how="left")

# Flag suspect matches:
# - Any multi-family PMC (>= 5 properties) with <= 4 units is likely a bad match
# - Pass 3 (proximity) matches are inherently lower quality
result = result.with_columns(
    pl.when(
        (pl.col("pmc_total_properties") >= 5)
        & pl.col("total_units").is_not_null()
        & (pl.col("total_units") <= 4)
    )
    .then(pl.lit("low"))
    .when(
        pl.col("_match_pass").is_not_null() & (pl.col("_match_pass") == 3)
    )
    .then(pl.lit("pass3"))
    .when(pl.col("APN").is_null())
    .then(pl.lit("unmatched"))
    .otherwise(pl.lit("high"))
    .alias("confidence"),
)

# Null out unit counts for low-confidence matches
result = result.with_columns(
    pl.when(pl.col("confidence") == "low")
    .then(None)
    .otherwise(pl.col("total_units"))
    .alias("total_units"),
    pl.when(pl.col("confidence") == "low")
    .then(None)
    .otherwise(pl.col("assessed_value"))
    .alias("assessed_value"),
)

result = result.drop("pmc_total_properties", "_match_pass")

result.write_csv(DATA_DIR / "pmc_properties_enriched.csv")

high = result.filter(pl.col("confidence") == "high")
low = result.filter(pl.col("confidence") == "low")
pass3 = result.filter(pl.col("confidence") == "pass3")
unmatched = result.filter(pl.col("confidence") == "unmatched")

print(f"Wrote {result.shape[0]:,} properties to pmc_properties_enriched.csv")
print(f"  High confidence:   {high.shape[0]:>4}  ({high['total_units'].sum():,} units)")
print(f"  Pass 3 (proximity):{pass3.shape[0]:>4}  (matched by nearest address)")
print(f"  Low confidence:    {low.shape[0]:>4}  (units nulled out, suspect parcel match)")
print(f"  Unmatched:         {unmatched.shape[0]:>4}  (no parcel found)")
print()
result.head(10)

Wrote 639 properties to pmc_properties_enriched.csv
  From apartments.com: 465
  From direct scrapes: 174
  Matched to parcels:  583
  Unmatched:           56



pmc_name,name,street_address,city,zip5,source,APN,total_units,assessed_value,landuse_label,SITUS_COMM
str,str,str,str,str,str,str,i64,i64,str,str
"""AMC""","""Artisan""","""1601 Broadway""","""San Diego""","""92101""","""apartments.com""","""5343600500""",265,109752000,null,"""SAN DIEGO"""
"""AMC""","""Echo Pointe""","""4300 Echo Ct""","""La Mesa""","""91941""","""apartments.com""","""4994810600""",4,1505583,"""Multifamily 2-4""","""LA MESA"""
"""AMC""","""Hillside Terrace Apartments""","""3262 College Pl""","""Lemon Grove""","""91945""","""apartments.com""","""4790623300""",150,15382829,"""Apartment 61+""","""LEMON GROVE"""
"""AMC""","""Lakeview Village""","""3115 Sweetwater Springs Blvd""","""Spring Valley""","""91977""","""apartments.com""",null,null,null,null,null
"""AMC""","""Outlook at Mission Valley""","""8360 Phyllis Pl""","""San Diego""","""92123""","""apartments.com""","""6773603300""",2,8361307,null,"""SAN DIEGO"""
"""AMC""","""The Commodore""","""200 E 31st St""","""National City""","""91950""","""apartments.com""","""5622524500""",92,22880465,"""Apartment 61+""","""NATIONAL CITY"""
"""AMC""","""Trolley Palm""","""4302 Palm Ave""","""La Mesa""","""91941""","""apartments.com""","""4990203700""",75,24084114,"""Apartment 61+""","""LA MESA"""
"""Alfred""","""Winslow""","""4353 Park Blvd""","""San Diego""","""92103""","""apartments.com""","""4453102100""",379,183151719,null,"""SAN DIEGO"""
"""Allen Properties""","""Courtyard on 68th""","""4823 68th St""","""San Diego""","""92115""","""apartments.com""","""4682602400""",38,13146625,"""Apartment 16-60""","""SAN DIEGO"""


## 8. Analysis

### PMCs by Number of Properties

In [9]:
pmc_by_props = (
    result
    .group_by("pmc_name")
    .agg(
        pl.len().alias("properties"),
        pl.col("total_units").sum().alias("total_units"),
        pl.col("assessed_value").sum().alias("total_assessed_value"),
        pl.col("city").n_unique().alias("cities"),
    )
    .sort("properties", descending=True)
)

print("Property management companies ranked by number of properties:")
pmc_by_props

Property management companies ranked by number of properties:


pmc_name,properties,total_units,total_assessed_value,cities
str,u32,i64,i64,u32
"""CONAM Management Corporation""",128,8341,2502879647,15
"""Greystar""",114,16572,5416538262,12
"""R.A. Snyder Properties, Inc.""",38,1923,325124717,9
"""Sunrise Management""",29,1748,282395831,6
"""Good Life Property Management""",26,878,457531058,9
…,…,…,…,…
"""Allen Properties""",1,38,13146625,1
"""J.F. Shea Co., Inc.""",1,159,28408088,1
"""Walsh & Company""",1,49,13386557,1


In [10]:
pmc_units = (
    result
    .filter(pl.col("total_units").is_not_null())
    .group_by("pmc_name")
    .agg(
        pl.len().alias("properties"),
        pl.col("total_units").sum().alias("total_units"),
        pl.col("assessed_value").sum().alias("total_assessed_value"),
        pl.col("total_units").mean().round(0).alias("avg_units_per_property"),
    )
    .sort("total_units", descending=True)
)

print("Top property management companies by total units managed:")
pmc_units.head(25)

Top property management companies by total units managed:


pmc_name,properties,total_units,total_assessed_value,avg_units_per_property
str,u32,i64,i64,f64
"""Greystar""",95,16572,5416538262,174.0
"""CONAM Management Corporation""",117,8341,2502879647,71.0
"""Garden Communities California""",6,2086,619476256,348.0
"""Essex""",8,2052,543027083,256.0
"""R.A. Snyder Properties, Inc.""",36,1923,325124717,53.0
…,…,…,…,…
"""Delta Property Management""",10,611,57112058,61.0
"""FairGrove Property Management""",14,609,121996211,44.0
"""AMC""",6,588,181966298,98.0


In [11]:
landuse_dist = (
    result
    .filter(pl.col("landuse_label").is_not_null())
    .group_by("landuse_label")
    .agg(
        pl.len().alias("properties"),
        pl.col("total_units").sum().alias("total_units"),
    )
    .sort("total_units", descending=True)
)

print("PMC-managed properties by land use type:")
landuse_dist

condo_props = result.filter(pl.col("landuse_label") == "Condo")
print(f"\nNote: {condo_props.shape[0]} properties are condos. Unit counts for condos")
print(f"may be inflated if the PMC manages only some units in the building.")

PMC-managed properties by land use type:


landuse_label,properties,total_units
str,u32,i64
"""Apartment 61+""",237,43426
"""Apartment 16-60""",153,6955
"""Condo""",33,1768
"""Hotel/Motel""",5,562
"""Single Family""",24,451
"""Apartment 5-15""",26,283
"""Multifamily 2-4""",4,21


## 9. Summary

In [12]:
high_conf = result.filter(pl.col("confidence") == "high")
low_conf = result.filter(pl.col("confidence") == "low")
no_match = result.filter(pl.col("confidence") == "unmatched")

print("=" * 65)
print("SAN DIEGO PMC-MANAGED RENTAL PROPERTIES")
print("=" * 65)
print(f"Property management companies:   {result['pmc_name'].n_unique():>8}")
print(f"Total properties (buildings):    {result.shape[0]:>8}")
print(f"  From apartments.com:           {result.filter(pl.col('source') == 'apartments.com').shape[0]:>8}")
print(f"  From direct scrapes:           {result.filter(pl.col('source') == 'direct').shape[0]:>8}")
print()
print(f"Data quality:")
print(f"  High confidence (reliable):    {high_conf.shape[0]:>8}  ({high_conf['total_units'].sum():,} units)")
print(f"  Low confidence (suspect):      {low_conf.shape[0]:>8}  (units unknown, bad parcel match)")
print(f"  Unmatched (no parcel found):   {no_match.shape[0]:>8}  (newer construction)")
print("-" * 65)
print()
print("Top 10 PMCs (high-confidence units only):")
for row in pmc_by_props.head(10).to_dicts():
    units = f"{row['total_units']:>7,}" if row.get('total_units') else '      ?'
    print(f"  {row['pmc_name']:<40} {row['properties']:>4} properties  {units} units")
print("=" * 65)


SAN DIEGO PMC-MANAGED RENTAL PROPERTIES
Property management companies:         74
Total properties (buildings):         639
  From apartments.com:                465
  From direct scrapes:                174
Matched to parcels:                   583
Total units (matched only):        59,890
Total assessed value:          $17,567,017,761
-----------------------------------------------------------------

Top 10 PMCs by properties:
  CONAM Management Corporation              128 properties    8,341 units
  Greystar                                  114 properties   16,572 units
  R.A. Snyder Properties, Inc.               38 properties    1,923 units
  Sunrise Management                         29 properties    1,748 units
  Good Life Property Management              26 properties      878 units
  Torrey Pines Property Management           19 properties      945 units
  Hoban Property  Management                 15 properties      997 units
  Hanken Cono Assad & Co., Inc.              14 p